In [ ]:
! pip install git+https://github.com/huggingface/transformers evaluate jiwer -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 22.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!unzip /content/drive/MyDrive/FYP/Pakistani_English_Scripted_Speech_Corpus_Daily_Use_Sentence.zip -d /content/drive/MyDrive/FYP

Archive:  /content/drive/MyDrive/FYP/Pakistani_English_Scripted_Speech_Corpus_Daily_Use_Sentence.zip
  inflating: /content/drive/MyDrive/FYP/README.txt  
  inflating: /content/drive/MyDrive/FYP/SPKINFO.txt  
  inflating: /content/drive/MyDrive/FYP/UTTERANCEINFO.txt  
   creating: /content/drive/MyDrive/FYP/WAV/
   creating: /content/drive/MyDrive/FYP/WAV/G0001/
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0101.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0102.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0103.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0104.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0105.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0106.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0107.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0108.wav  
  inflating: /content/drive/MyDrive/FYP/WAV/G0001/G0001_0_S0109.wav  
  infl

In [ ]:
import os
import time
import re
import pandas as pd
import csv
from evaluate import load
from transformers import pipeline
from transformers.models.whisper.english_normalizer import BasicTextNormalizer


In [ ]:
# Define the path to the main directory containing the subfolders
main_directory = "/content/drive/MyDrive/FYP-Language_learning/pakistani_english_dataset/WAV"

# List all subfolders inside the main directory
subfolders = [f.path for f in os.scandir(main_directory) if f.is_dir()]

# Initialize a list to store the full paths of .wav files
wav_files = []

# Loop over each subfolder
for subfolder in subfolders:
    # Get the list of all files in the current subfolder
    wav_files_in_subfolder = [os.path.join(subfolder, file) for file in os.listdir(subfolder) if file.endswith('.wav')]

    wav_files.extend(wav_files_in_subfolder[:10])


In [ ]:
print("Length of sample data : ", len(wav_files))

Length of sample data :  140


In [ ]:
data = pd.read_csv("/content/drive/MyDrive/FYP-Language_learning/pakistani_english_dataset/UTTERANCEINFO.txt", delimiter='\t')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2191 entries, 0 to 2190
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   CHANNEL        2191 non-null   object
 1   UTTRANS_ID     2191 non-null   object
 2   SPEAKER_ID     2191 non-null   object
 3   PROMPT         2191 non-null   object
 4   TRANSCRIPTION  2191 non-null   object
dtypes: object(5)
memory usage: 85.7+ KB


In [ ]:
data = data[["UTTRANS_ID", "TRANSCRIPTION"]]

In [ ]:
orignal_transcriptions = {}

# Loop through each .wav file and get the corresponding transcription from the DataFrame
for wav_path in wav_files:
  wav_id = os.path.basename(wav_path)
  transcription = data.loc[data['UTTRANS_ID'] == wav_id, 'TRANSCRIPTION'].iloc[0]
  orignal_transcriptions[wav_id] = transcription


# **Whisper distil large v3**

In [ ]:
# Use a pipeline as a high-level helper
pipe = pipeline("automatic-speech-recognition", model="distil-whisper/distil-large-v3")

Device set to use cpu


In [ ]:
def transcribe_speech(filepath):
    output = pipe(
        filepath,
        max_new_tokens=256,
        generate_kwargs={
             "task": "transcribe"
        #     "language": "english",
        },
        chunk_length_s=30,
        batch_size=8,
    )
    return output["text"]

In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    # Call the transcribe function to get the transcription for each wav file
    transcription = transcribe_speech(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `

KeyboardInterrupt: 

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")
# 1 hour 35 minute to transcribe 109 audios

In [ ]:
len(predicted_transcriptions)

108

In [ ]:
# Output CSV file path
output_csv_file = '/content/drive/MyDrive/FYP/transcriptions_comparison.csv'

# Open the CSV file for writing
with open(output_csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)

    # Write header row
    writer.writerow(['id', 'original_transcription', 'predicted_transcription'])

    # Iterate over the ids in the original transcription dictionary
    for id, original in predicted_transcriptions.items():
        # Get the predicted transcription using the id
        predicted = orignal_transcriptions.get(id)

        # Write the row for the current id, original transcription, and predicted transcription
        writer.writerow([id, original, predicted])

print(f"CSV file '{output_csv_file}' has been created successfully.")


CSV file '/content/drive/MyDrive/FYP/transcriptions_comparison.csv' has been created successfully.


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/FYP/transcriptions_comparison.csv')
df['predicted_transcription'] = df['predicted_transcription'].str.strip()

In [ ]:
normalizer = BasicTextNormalizer()

df['normalized_transcription'] = df['predicted_transcription'].apply(normalizer).str.strip()

In [ ]:
wer_metric = load("wer")

wer_ortho = 100 * wer_metric.compute(
    references=df['original_transcription'], predictions=df['normalized_transcription'])

wer_ortho

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


46.09826589595375

In [ ]:
# Initialize the WER metric
wer_metric = load("wer")

# Function to check if a transcription contains digits
def contains_digits(text):
    # Return True if the text contains any digits
    return bool(re.search(r'\d', text))

# Step 1: Filter out rows where 'normalized_transcription' contains any digits
df_cleaned = df[~df['normalized_transcription'].apply(contains_digits)]

# Step 2: Calculate WER on the cleaned data
wer_ortho = 100 * wer_metric.compute(
    references=df_cleaned['original_transcription'], predictions=df_cleaned['normalized_transcription'])

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 13.136094674556212%


In [ ]:
df[['normalized_transcription', 'original_transcription']].iloc[60:80,:]

,normalized_transcription,original_transcription
60,i was just wondering if you happen to know any...,i was just wondering if you happen to know any...
61,it is a day to help the needy,it is a day to help the needy
62,675508 8 8104 671463,six seven five zero five one zero eight eight ...
63,51230736840304304 6702,five one two three zero seven three six eight ...
64,and trash say bad things about your opponent,and try say bad things about your opponent
65,3875662260 2 6033 088451,three eight seven five six six two two six zer...
66,5782370718804 825,five seven eight two three six seven zero seve...
67,768402 6268134 7831,seven six eight four zero two six two six eigh...
68,wow i just ordered from amazon is this to be e...,wow i just ordered from amzon is this to be ex...
69,think about what you are going to see and be y...,think about what you're going to say add we ar...


# **Wav2vec2**

In [ ]:
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-100h")

config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/wav2vec2-base-100h were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.mask_time_emb_vector']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-100h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    transcription = pipe(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

In [ ]:
predicted_transcriptions

{'G0007_0_S0020.wav': {'text': 'GIVE UP STEPS THAT HINDER NORMAR DEVELOPMENT IN RELATIONS'},
 'G0007_0_S0023.wav': {'text': "THE GARDS SOLER BANNA TR YEAR'S ELECTRICITY FROM SOM LIGHT"},
 'G0007_0_S0031.wav': {'text': 'TA RUBELL ROS ALVAS BURSTLY THE KING OF ROKINROD'},
 'G0007_0_S0005.wav': {'text': 'SEVON FOUR SEVON SIX FIVE THREE SIX DEO FOR THREE THREE ZARO TWO SIX ONE SEVON EIGHTH ONE'},
 'G0007_0_S0006.wav': {'text': 'FOUR SAON SIX EIGHTH TO ONE EIGTH LIDO DU FIVE DU THEOR THREE SIX FIVE TEE O FOUR FOUR SIX'},
 'G0007_0_S0024.wav': {'text': 'AND IC TODI OF CHEF JOSEPH OF THA NESBERSS INDIANS'},
 'G0007_0_S0040.wav': {'text': 'HANGS FOR PLAYING ALONE I JUST HAVE TO SIT FOR A WINE'},
 'G0007_0_S0043.wav': {'text': 'MONE PESSENGER DIED AFTER BEING BARSHLY OUT OF THE PLAIN'},
 'G0007_0_S0018.wav': {'text': 'NO NO I HAVE TO GO NOW I HAVE TO SEE THE PHROZEN JARS NOW TO NIGHT'},
 'G0007_0_S0042.wav': {'text': 'AND SHE TANKED E FRANCH PEAPER FOR SHIRING HIGH DESPAIR'},
 'G0007_0_S0013.wa

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")

Total time taken for transcription: 510.97 seconds


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/FYP/transcriptions_comparison.csv')
df.head()

,id,predicted_transcription,original_transcription,predicted_transcription_wav2vec2
0,G0007_0_S0020.wav,Give up steps that hinder normal development ...,give up steps that hinder normal development i...,GIVE UP STEPS THAT HINDER NORMAR DEVELOPMENT I...
1,G0007_0_S0023.wav,The car's solar panel creates electricity fro...,the car's solar panel creates electricity from...,THE GARDS SOLER BANNA TR YEAR'S ELECTRICITY FR...
2,G0007_0_S0031.wav,"The rebel was Elvis Persley, the king of rock...",the rebel was elvis presley the king of rock a...,TA RUBELL ROS ALVAS BURSTLY THE KING OF ROKINROD
3,G0007_0_S0005.wav,747-76-53604-3-30-26-16-1781,seven four seven six five three six zero four ...,SEVON FOUR SEVON SIX FIVE THREE SIX DEO FOR TH...
4,G0007_0_S0006.wav,4768-18-0252-0-0352-0365-04-46,four seven six eight one eight zero two five t...,FOUR SAON SIX EIGHTH TO ONE EIGTH LIDO DU FIVE...


In [ ]:

# Step 2: Create a new column 'transcription' based on the 'id' column matching dictionary keys
df['predicted_transcription_wav2vec2'] = df['id'].map(lambda x: predicted_transcriptions.get(x, {}).get('text', 'No transcription found'))

#df['predicted_transcription_wav2vec2'] = df['predicted_transcription_wav2vec2'].str.strip()

# Step 3: Save the updated DataFrame back to the CSV file (overwrite the original file)
#df.to_csv('/content/drive/MyDrive/FYP/transcriptions_comparison.csv', index=False) Done


In [ ]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

df['normalized_transcription_wav2vec2'] = df['predicted_transcription_wav2vec2'].apply(normalizer)

In [ ]:
# Initialize the WER metric
wer_metric = load("wer")

wer_ortho = 100 * wer_metric.compute(
    references=df['original_transcription'], predictions=df['normalized_transcription_wav2vec2'])

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 39.667630057803464%


In [ ]:
df[['normalized_transcription_wav2vec2', 'original_transcription']].iloc[1:20,:]

,normalized_transcription_wav2vec2,original_transcription
1,the gards soler banna tr year s electricity fr...,the car's solar panel creates electricity from...
2,ta rubell ros alvas burstly the king of rokinrod,the rebel was elvis presley the king of rock a...
3,sevon four sevon six five three six deo for th...,seven four seven six five three six zero four ...
4,four saon six eighth to one eigth lido du five...,four seven six eight one eight zero two five t...
5,and ic todi of chef joseph of tha nesberss ind...,and its story of jeff joseph of the nez perce ...
6,hangs for playing alone i just have to sit for...,thanks for playing along i just have to sit fo...
7,mone pessenger died after being barshly out of...,one passenger died after being partially out o...
8,no no i have to go now i have to see the phroz...,now now i have to go now i have to see the fro...
9,and she tanked e franch peaper for shiring hig...,and she thanked the french people for sharing ...
10,i laughed so hard so you made me coke on the f...,i laughed so hard so you made me choke on the ...


**Wav2vec2N-gram**

In [ ]:
!pip install pyctcdecode pypi-kenlm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.4/278.4 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.8/471.8 kB 22.4 MB/s eta 0:00:00
  Created wheel for pypi-kenlm: filename=pypi_kenlm-0.1.20220713-cp310-cp310-linux_x86_64.whl size=2935996 sha256=0ca94c0bcd8bfd0588eb4ff4f27a69cb5a739b521842a8025e147efaa5436cda
  Stored in directory: /root/.cache/pip/wheels/24/6c/ca/e305540f351405de41fe750ab134480a3646f3eff3f441d4f6
Successfully built pypi-kenlm


In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="patrickvonplaten/wav2vec2-base-100h-with-lm")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

alphabet.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

language_model/attrs.json:   0%|          | 0.00/78.0 [00:00<?, ?B/s]

language_model/unigrams.txt:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

4-gram.bin:   0%|          | 0.00/3.12G [00:00<?, ?B/s]

In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    transcription = pipe(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")

Total time taken for transcription: 421.40 seconds


In [ ]:

# Step 2: Create a new column 'transcription' based on the 'id' column matching dictionary keys
df['predicted_transcription_wav2vec2Ngram'] = df['id'].map(lambda x: predicted_transcriptions.get(x, {}).get('text', 'No transcription found'))

#df['predicted_transcription_wav2vec2'] = df['predicted_transcription_wav2vec2'].str.strip()

# Step 3: Save the updated DataFrame back to the CSV file (overwrite the original file)
#df.to_csv('/content/drive/MyDrive/FYP/transcriptions_comparison.csv', index=False)


In [ ]:
df.head()

,id,predicted_transcription,original_transcription,predicted_transcription_wav2vec2,normalized_transcription_wav2vec2,predicted_transcription_wav2vec2Ngram
0,G0007_0_S0020.wav,Give up steps that hinder normal development ...,give up steps that hinder normal development i...,GIVE UP STEPS THAT HINDER NORMAR DEVELOPMENT I...,give up steps that hinder normar development i...,GIVE UP STEPS THAT HINDER NORMAL DEVELOPMENT I...
1,G0007_0_S0023.wav,The car's solar panel creates electricity fro...,the car's solar panel creates electricity from...,THE GARDS SOLER BANNA TR YEAR'S ELECTRICITY FR...,the gards soler banna tr year s electricity fr...,THE GUARDS SOLER BANNER THREE YEARS ELECTRICIT...
2,G0007_0_S0031.wav,"The rebel was Elvis Persley, the king of rock...",the rebel was elvis presley the king of rock a...,TA RUBELL ROS ALVAS BURSTLY THE KING OF ROKINROD,ta rubell ros alvas burstly the king of rokinrod,THE REBEL ROSE ALVA BURSTLY THE KING OF ROKENROD
3,G0007_0_S0005.wav,747-76-53604-3-30-26-16-1781,seven four seven six five three six zero four ...,SEVON FOUR SEVON SIX FIVE THREE SIX DEO FOR TH...,sevon four sevon six five three six deo for th...,SEVEN FOUR SEVEN SIX FIVE THREE SIX ZERO FOUR ...
4,G0007_0_S0006.wav,4768-18-0252-0-0352-0365-04-46,four seven six eight one eight zero two five t...,FOUR SAON SIX EIGHTH TO ONE EIGTH LIDO DU FIVE...,four saon six eighth to one eigth lido du five...,FOR SALMON SIX EIGHT TO ONE EIGHTH LIDODU FIVE...


In [ ]:
normalizer = BasicTextNormalizer()

df['normalized_transcription_wav2vec2Ngram'] = df['predicted_transcription_wav2vec2Ngram'].apply(normalizer)

In [ ]:
# Initialize the WER metric
wer_metric = load("wer")

wer_ortho = 100 * wer_metric.compute(
    references=df['original_transcription'], predictions=df['normalized_transcription_wav2vec2Ngram'])

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 26.300578034682083%


In [ ]:
df[['normalized_transcription_wav2vec2Ngram', 'original_transcription']].iloc[50:70,:]

,normalized_transcription_wav2vec2Ngram,original_transcription
50,jot she is unique and the music word needs mor...,truth she is unique and the music world needs ...
51,two three three two seven four five four eight...,two three three two seven four five four eight...
52,yes you can you can spoil a perfectly nice dance,yes you can you can spoil a perfectly nice dance
53,did you know that lesly man is actually his wi...,did you know that leslie mann is actually his ...
54,zero four one zelo three seven one seven at th...,zero four one zero three seven one seven eight...
55,a god dis jes at a tripped store for one dollar,i got this dress at a thrift store for one dollar
56,it had its good points still delights saber bu...,it had it's good points though the light saber...
57,it was replaced by a store and visitors center...,it was replaced by a store and visitors center...
58,one two three at six to seven for seven zero s...,one two three eight six two seven four seven z...
59,have cost a per cent fall in the value of the ...,have caused a percent fall in the value of the...


# **Whiser distil small en**

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="distil-whisper/distil-small.en")

config.json:   0%|          | 0.00/2.26k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/332M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/282k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/999k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.17k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
def transcribe_speech(filepath):
    output = pipe(
        filepath,
        max_new_tokens=256,
        chunk_length_s=30,
        batch_size=8,
    )
    return output["text"]

In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    # Call the transcribe function to get the transcription for each wav file
    transcription = transcribe_speech(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a 

In [ ]:
predicted_transcriptions

{'G0007_0_S0020.wav': ' Give up steps that hinder normal development in relations.',
 'G0007_0_S0023.wav': " The car's solar panel creates electricity from sunlight.",
 'G0007_0_S0031.wav': " The rebel was Elvis Persley, the king of Rockin' Road.",
 'G0007_0_S0005.wav': ' 7476560433030261781',
 'G0007_0_S0006.wav': ' 4-7-6-8-1-8-0-2-2-3-6-5-044-6',
 'G0007_0_S0024.wav': ' and its story of Chief Joseph of the Naspurs Indians.',
 'G0007_0_S0040.wav': ' Thanks for playing along. I just have to sit for a while.',
 'G0007_0_S0043.wav': ' One passenger died after being partially out of the plane.',
 'G0007_0_S0018.wav': ' Now, now I have to go now. I have to see the frozen Charles now tonight.',
 'G0007_0_S0042.wav': ' And she thanked the French people for sharing her despair.',
 'G0007_0_S0013.wav': ' I laughed so hard so you made me joke on the food I am eating.',
 'G0007_0_S0027.wav': ' I am a fresh grad with no relative experience so yes, Lawl.',
 'G0007_0_S0025.wav': ' The goal is for p

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")


Total time taken for transcription: 815.24 seconds


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/FYP/transcriptions_comparison.csv')

In [ ]:

# Step 2: Create a new column 'transcription' based on the 'id' column matching dictionary keys
df['predicted_transcription_Whisper-med-en'] = df['id'].map(predicted_transcriptions)

df['predicted_transcription_Whisper-med-en'] = df['predicted_transcription_Whisper-med-en'].str.strip()

# Step 3: Save the updated DataFrame back to the CSV file (overwrite the original file)
#df.to_csv('/content/drive/MyDrive/FYP/transcriptions_comparison.csv', index=False)


In [ ]:
normalizer = BasicTextNormalizer()

df['normalized_predicted_transcription_Whisper-med-en'] = df['predicted_transcription_Whisper-med-en'].apply(normalizer)

In [ ]:
df.head()

,id,predicted_transcription,original_transcription,predicted_transcription_wav2vec2,normalized_transcription_wav2vec2,predicted_transcription_wav2vec2Ngram,predicted_transcription_Whisper-med-en,normalized_predicted_transcription_Whisper-med-en
0,G0007_0_S0020.wav,Give up steps that hinder normal development ...,give up steps that hinder normal development i...,GIVE UP STEPS THAT HINDER NORMAR DEVELOPMENT I...,give up steps that hinder normar development i...,GIVE UP STEPS THAT HINDER NORMAL DEVELOPMENT I...,Give up steps that hinder normal development i...,give up steps that hinder normal development i...
1,G0007_0_S0023.wav,The car's solar panel creates electricity fro...,the car's solar panel creates electricity from...,THE GARDS SOLER BANNA TR YEAR'S ELECTRICITY FR...,the gards soler banna tr year s electricity fr...,THE GUARDS SOLER BANNER THREE YEARS ELECTRICIT...,The car's solar panel creates electricity from...,the car s solar panel creates electricity from...
2,G0007_0_S0031.wav,"The rebel was Elvis Persley, the king of rock...",the rebel was elvis presley the king of rock a...,TA RUBELL ROS ALVAS BURSTLY THE KING OF ROKINROD,ta rubell ros alvas burstly the king of rokinrod,THE REBEL ROSE ALVA BURSTLY THE KING OF ROKENROD,"The rebel was Elvis Persley, the king of Rocki...",the rebel was elvis persley the king of rockin...
3,G0007_0_S0005.wav,747-76-53604-3-30-26-16-1781,seven four seven six five three six zero four ...,SEVON FOUR SEVON SIX FIVE THREE SIX DEO FOR TH...,sevon four sevon six five three six deo for th...,SEVEN FOUR SEVEN SIX FIVE THREE SIX ZERO FOUR ...,7476560433030261781,7476560433030261781
4,G0007_0_S0006.wav,4768-18-0252-0-0352-0365-04-46,four seven six eight one eight zero two five t...,FOUR SAON SIX EIGHTH TO ONE EIGTH LIDO DU FIVE...,four saon six eighth to one eigth lido du five...,FOR SALMON SIX EIGHT TO ONE EIGHTH LIDODU FIVE...,4-7-6-8-1-8-0-2-2-3-6-5-044-6,4 7 6 8 1 8 0 2 2 3 6 5 044 6


In [ ]:
# Initialize the WER metric
wer_metric = load("wer")

# Function to check if a transcription contains digits
def contains_digits(text):
    # Return True if the text contains any digits
    return bool(re.search(r'\d', text))

# Step 1: Filter out rows where 'normalized_transcription' contains any digits
df_cleaned = df[~df['normalized_predicted_transcription_Whisper-med-en'].apply(contains_digits)]

# Step 2: Calculate WER on the cleaned data
wer_ortho = 100 * wer_metric.compute(
    references=df_cleaned['original_transcription'], predictions=df_cleaned['normalized_predicted_transcription_Whisper-med-en'])

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 14.441747572815533%


**Whisper Samll en**

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-small.en")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
def transcribe_speech(filepath):
    output = pipe(
        filepath,
        max_new_tokens=256,
        chunk_length_s=30,
        batch_size=8,
    )
    return output["text"]

In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    # Call the transcribe function to get the transcription for each wav file
    transcription = transcribe_speech(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a 

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")


Total time taken for transcription: 814.56 seconds


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/FYP-Language_learning/transcriptions_comparison.csv")

In [ ]:
# Step 2: Create a new column 'transcription' based on the 'id' column matching dictionary keys
df['predicted_transcription_Whisper-small-en'] = df['id'].map(predicted_transcriptions)

df['predicted_transcription_Whisper-small-en'] = df['predicted_transcription_Whisper-small-en'].str.strip()


In [ ]:
# Helper function to check if a transcription contains digits
def contains_digits(text):
    return bool(re.search(r'\d', text))  # Returns True if there are any digits

# Initialize the normalizer
normalizer = BasicTextNormalizer()

# Dictionary to store normalized transcriptions without digits
normalized_dict = {}

# Filter and remove transcriptions with digits from the original dictionary
for audio_id, text in list(predicted_transcriptions.items()):  # Iterate through original dictionary
    if contains_digits(text):  # If transcription contains digits, remove from original dict
        del predicted_transcriptions[audio_id]  # Remove this entry from the original dictionary
        del orignal_transcriptions[audio_id]
    else:
        # Normalize the transcription if it does not contain digits
        normalized_text = normalizer(text.strip())
        normalized_dict[audio_id] = normalized_text  # Add to normalized dictionary

print("Length of normalized dict",len(normalized_dict))
print("Length of original dict",len(orignal_transcriptions))

Length of normalized dict 54
Length of original dict 54


In [ ]:
# from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# normalizer = BasicTextNormalizer()

# # Apply the normalizer to each transcription in the dictionary
# normalized_dict = {audio_id: normalizer(text.strip()) for audio_id, text in predicted_transcriptions.items()}

# # Print the normalized transcriptions
print(normalized_dict)

{'G0007_0_S0020.wav': 'give up steps that hinder normal development in relations ', 'G0007_0_S0023.wav': 'the car s solar panel creates electricity from sunlight ', 'G0007_0_S0031.wav': 'the rebel was elvis presley the king of rock and roll ', 'G0007_0_S0024.wav': 'and its story of chief joseph of the naspers indians', 'G0007_0_S0040.wav': 'thanks for playing along i just have to sit for a while ', 'G0007_0_S0043.wav': 'one passenger died after being partially out of the plane ', 'G0007_0_S0018.wav': 'now now i have to go now i have to see the frozen charles now tonight ', 'G0007_0_S0042.wav': 'and she thanked the french people for sharing her despair ', 'G0008_0_S0049.wav': 'who do you think the wolves and the mavericks will take ', 'G0008_0_S0047.wav': 'and no it s not as if any of her already early stuff was less provocative ', 'G0008_0_S0016.wav': 'she looked down her nose at me and sneered ', 'G0008_0_S0043.wav': 'yes that s it i liked that one what about benny goodman you mention

In [ ]:
orignal_transcriptions

{'G0007_0_S0020.wav': 'give up steps that hinder normal development in relations',
 'G0007_0_S0023.wav': "the car's solar panel creates electricity from sunlight",
 'G0007_0_S0031.wav': 'the rebel was elvis presley the king of rock and roll',
 'G0007_0_S0024.wav': 'and its story of jeff joseph of the nez perce indians',
 'G0007_0_S0040.wav': 'thanks for playing along i just have to sit for a while',
 'G0007_0_S0043.wav': 'one passenger died after being partially out of the plane',
 'G0007_0_S0018.wav': 'now now i have to go now i have to see the frozen charles now tonight',
 'G0007_0_S0042.wav': 'and she thanked the french people for sharing her despair',
 'G0008_0_S0049.wav': 'who do you think the wolves and the mavericks will take',
 'G0008_0_S0047.wav': "i know it's not as if any of her really early stuff was less provocative",
 'G0008_0_S0016.wav': 'she looked down her nose at me and sneered',
 'G0008_0_S0043.wav': "yes that's it i liked that one what about benny goodman you mentio

In [ ]:
# Initialize the WER metric
wer_metric = load("wer")

# Step 2: Calculate WER on the cleaned data
wer_ortho = 100 * wer_metric.compute(
    references=list(orignal_transcriptions.values()), predictions=list(normalized_dict.values()))

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 12.864493996569468%


**Whisper Base en**

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")

config.json:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    # Call the transcribe function to get the transcription for each wav file
    transcription = transcribe_speech(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `

In [ ]:
len(predicted_transcriptions)

140

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")


Total time taken for transcription: 504.35 seconds


In [ ]:
# Step 2: Create a new column 'transcription' based on the 'id' column matching dictionary keys
df['predicted_transcription_Whisper-base-140-en'] = df['id'].map(predicted_transcriptions)

df['predicted_transcription_Whisper-base-140-en'] = df['predicted_transcription_Whisper-base-140-en'].str.strip()


In [ ]:
import re
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# Helper function to check if a transcription contains digits
def contains_digits(text):
    return bool(re.search(r'\d', text))  # Returns True if there are any digits

# Initialize the normalizer
normalizer = BasicTextNormalizer()

# Dictionary to store normalized transcriptions without digits
normalized_dict = {}

keys_to_remove = set(orignal_transcriptions.keys()) - set(predicted_transcriptions.keys())  # Find keys in original_dict but not in predicted_dict

# Remove those keys from original_dict
for key in keys_to_remove:
    del orignal_transcriptions[key]

# Filter and remove transcriptions with digits from the original dictionary
for audio_id, text in list(predicted_transcriptions.items()):  # Iterate through original dictionary
    if contains_digits(text):  # If transcription contains digits, remove from original dict
        del predicted_transcriptions[audio_id]  # Remove this entry from the original dictionary
        del orignal_transcriptions[audio_id]
    else:
        # Normalize the transcription if it does not contain digits
        normalized_text = normalizer(text.strip())
        normalized_dict[audio_id] = normalized_text  # Add to normalized dictionary

print("Length of normalized dict",len(normalized_dict))
print("Length of original dict",len(orignal_transcriptions))

Length of normalized dict 99
Length of original dict 99


In [ ]:
print(normalized_dict)
print(orignal_transcriptions)

{'G0007_0_S0020.wav': 'give up steps that hinder normal development in relations ', 'G0007_0_S0023.wav': 'the car s solar panel creates electricity from sunlight ', 'G0007_0_S0031.wav': 'the rebel was always personally the king of rock and roll ', 'G0007_0_S0024.wav': 'and its story of chief joseph of the naspers indians ', 'G0007_0_S0040.wav': 'thanks for playing along i just have to sit for a while ', 'G0007_0_S0043.wav': 'one person s here tight after being partially out of the plane ', 'G0007_0_S0018.wav': 'now now i have to go now i have to see the frozen charles now tonight ', 'G0007_0_S0042.wav': 'and she thanked the french people for sharing her despair ', 'G0008_0_S0049.wav': 'who do you think the walls and the mavericks will take ', 'G0008_0_S0047.wav': 'and now it s not as if any of her really early stuff was less provocative ', 'G0008_0_S0016.wav': 'she looked down her nose at me and sneered ', 'G0008_0_S0043.wav': 'yes that s it i like that one what about benny goodman you

In [ ]:
import re
from evaluate import load

# Initialize the WER metric
wer_metric = load("wer")

# Step 2: Calculate WER on the cleaned data
wer_ortho = 100 * wer_metric.compute(
    references=list(orignal_transcriptions.values()), predictions=list(normalized_dict.values()))

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 17.11378353376503%


**Whisper distil Medium en**

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-medium.en")

config.json:   0%|          | 0.00/1.95k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/1.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# Start the timer
start_time = time.time()

# Initialize dictionary to store the predicted transcriptions
predicted_transcriptions = {}

# Loop over each .wav file in wav_files
for wav_file in wav_files:
    # Call the transcribe function to get the transcription for each wav file
    transcription = transcribe_speech(wav_file)

    # Get the file name (ID) from the full path
    wav_id = os.path.basename(wav_file)

    # Store the transcription in the dictionary
    predicted_transcriptions[wav_id] = transcription

# Stop the timer after processing all files
end_time = time.time()

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/automatic_speech_recognition.py:312: FutureWarning: `max_new_tokens` is deprecated and will be removed in version 4.49 of Transformers. To remove this warning, pass `max_new_tokens` as a key inside `generate_kwargs` instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `

In [ ]:
# Calculate total time taken in seconds
total_time = end_time - start_time

# Print the total time taken
print(f"Total time taken for transcription: {total_time:.2f} seconds")


Total time taken for transcription: 2347.68 seconds


In [ ]:
# Step 2: Create a new column 'transcription' based on the 'id' column matching dictionary keys
df['predicted_transcription_Whisper-distill-medium-en'] = df['id'].map(predicted_transcriptions)

df['predicted_transcription_Whisper-distill-medium-en'] = df['predicted_transcription_Whisper-distill-medium-en'].str.strip()


In [ ]:
import re
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

# Helper function to check if a transcription contains digits
def contains_digits(text):
    return bool(re.search(r'\d', text))  # Returns True if there are any digits

# Initialize the normalizer
normalizer = BasicTextNormalizer()

# Dictionary to store normalized transcriptions without digits
normalized_dict = {}

keys_to_remove = set(orignal_transcriptions.keys()) - set(predicted_transcriptions.keys())  # Find keys in original_dict but not in predicted_dict

# Remove those keys from original_dict
for key in keys_to_remove:
    del orignal_transcriptions[key]

# Filter and remove transcriptions with digits from the original dictionary
for audio_id, text in list(predicted_transcriptions.items()):  # Iterate through original dictionary
    if contains_digits(text):  # If transcription contains digits, remove from original dict
        del predicted_transcriptions[audio_id]  # Remove this entry from the original dictionary
        del orignal_transcriptions[audio_id]
    else:
        # Normalize the transcription if it does not contain digits
        normalized_text = normalizer(text.strip())
        normalized_dict[audio_id] = normalized_text  # Add to normalized dictionary

print("Length of normalized dict",len(normalized_dict))
print("Length of original dict",len(orignal_transcriptions))

Length of normalized dict 53
Length of original dict 53


In [ ]:
import re
from evaluate import load

# Initialize the WER metric
wer_metric = load("wer")

# Step 2: Calculate WER on the cleaned data
wer_ortho = 100 * wer_metric.compute(
    references=list(orignal_transcriptions.values()), predictions=list(normalized_dict.values()))

print(f"Word Error Rate (WER): {wer_ortho}%")

Word Error Rate (WER): 12.390924956369982%
